# Ruse

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import shutil
import tempfile
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from helpers import *

## Default experiment configuration

In [ ]:
default_config = {
    "timeout": datetime.timedelta(hours=1),
    "max_iterations": 5,
    "max_sequence_size": 2,
    "max_mutations": 3,
    "max_memory_usage": "100GiB",
    "workers_count": 64
}

## Helper functions

#### Some notebook helper functions

In [ ]:
def display_df(df: pd.DataFrame):
    s = df.style.set_properties(**{'text-align': 'left'})
    s.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    html_table = s.to_html(justify='left', notebook=True, show_dimensions=True, index=False)

    scrollable_html = f"""
<div style="max-height: 300px; overflow: auto;">
    {html_table}
</div>
"""
    display(HTML(scrollable_html))

class StopExecution(Exception):
    def _render_traceback_(self):
        return []

#### Parse results

In [ ]:
def sort_by_oop_and_side_effects(df: pd.DataFrame):
    oop_category_order = {
        "Full OOP": 0,
        "Primitive Objects": 1,
        "Primitive": 2,
    }

    side_effects_order = {
        "With Side Effects": 0,
        "Possibly With Side Effects": 1,
        "Without Side Effects": 2
    }

    def sort_func(col):
        if col.name == "oop_category":
            return col.apply(lambda x: oop_category_order[x])
        if col.name == "side_effects":
            return col.apply(lambda x: side_effects_order[x])

    df.sort_values(by=["oop_category", "side_effects"], key=sort_func, inplace=True)

def sort_by_found_time(df: pd.DataFrame):
    def sort_func(col):
        if col.name == "found":
            return col.apply(lambda x: 0 if pd.notna(x) else 1)
        if col.name == "total_time":
            return col

    df.sort_values(by=["found", "total_time"], key=sort_func, inplace=True)

def create_tasks_dataframe(tasks, dry_run) -> pd.DataFrame:
    df = pd.DataFrame(tasks)
    df["name"] = df["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    df["total_time"] = df["total_time"].apply(lambda x: pd.Timedelta(seconds=x["secs"], nanoseconds=x["nanos"]).total_seconds() if pd.notnull(x) else x).astype(dtype="float64")
    df["path"] = df["path"].apply(lambda x: os.path.relpath(os.path.abspath(x), get_workspace_root()))
    df.rename(columns={"total_statistics": "stats"}, inplace=True)

    if "category" in df.columns:
        df["oop_category"] = df["category"].apply(lambda x: x.split(":")[0])
        df["side_effects"] = df["category"].apply(lambda x: x.split(":")[1])
        df.pop("category")

    if not dry_run:
        df = pd.json_normalize(df.to_dict(orient="records"))
        df["found.num_mutations"] = df["found.num_mutations"].astype(dtype="Int64")
        if "found" in df.columns:
            df.pop("found") 
        df.rename(columns={"found.found": "found"}, inplace=True)
        sort_by_found_time(df)

    ordered_columns = ['name',
                       'total_time',
                       'found',
                       'error',
                       'found.has_side_effects',
                       'found.num_mutations',
                       'path',
                       'source',
                       'category',
                       'oop_category',
                       'side_effects',
                       'opcode_count',
                       'max_vm_usage',
                       'stats.Evaluated',
                       'stats.BankSize',
                       'stats.FoundContextCount',
                       'stats.MaxMutatingOpcodes',
                       'stats.MaxDepth',
                       'stats.MaxSize',
                       'string_literals',
                       'num_literals',
                       'iterations']

    ordered_columns = [col for col in ordered_columns if col in df.columns]
    df = df[ordered_columns]

    return df

def parse_results(results_dir, dry_run):
    tasks = []
    metadata = None
    for file in os.listdir(results_dir):
        if file == 'metadata.json':
            with open(os.path.join(results_dir, file), "r") as f:
                metadata = json.load(f)
        elif file.startswith('task_') and file.endswith('.json'):
            with open(os.path.join(results_dir, file), "r") as f:
                tasks.append(json.load(f))
    df = create_tasks_dataframe(tasks, dry_run)
    metadata["tasks_count"] = len(df)
    metadata["total_time"] = datetime.timedelta(seconds=df["total_time"].sum())
    metadata["error_count"] = df["error"].notna().sum()
    return metadata, df

In [ ]:
class TaskParserError(Exception):
    def __init__(self, errors):
        self.errors = errors
        
    def _render_traceback_(self):
        return []

def get_all_tasks(tasks):
    with tempfile.TemporaryDirectory() as temp_dir:
        results_dir = os.path.join(temp_dir, "results")

        run_ruse(tasks, results_dir, dry_run=True, in_background=False)
        metadata, tasks = parse_results(results_dir, dry_run=True)

    if metadata["error_count"] > 0:
        print("Errors:")
        raise TaskParserError(tasks[tasks["error"].notna()][["name", "error"]])

    sort_by_oop_and_side_effects(tasks)
    return tasks

#### Run experiment helper function

In [ ]:
def run_experiment(name, tasks, in_background=False, **kwargs):
    os.makedirs("results", exist_ok=True)

    results_dir = os.path.join("results", f"{name}_results")
    log_file = os.path.join("results", f"{name}_log.jsonl")

    if os.path.exists(results_dir):
        shutil.rmtree(results_dir)

    config = default_config.copy()
    config.update(kwargs)

    run_ruse(tasks, results_dir, log_file=log_file, in_background=in_background, **config)
    if not in_background:
        return parse_results(results_dir, dry_run=False)
    else:
        return get_ruse_pid(results_dir)


## Experiments

### RQ1 - Can Ruse solve synthesis tasks from all categories correctly?

In [ ]:
tasks_paths = [
    os.path.join(get_workspace_root(), "tasks/benchmarks/"),
]

try:
    tasks = get_all_tasks(tasks_paths)

    display_df(tasks[["name", "oop_category", "side_effects"]])
    print(f'{len(tasks)} Tasks in total')
except Exception as e:
    if isinstance(e, TaskParserError):
        display_df(e.errors)
        raise StopExecution
    else: 
        raise e
    

In [ ]:
ruse_pid = run_experiment("ruse_all_tasks", tasks_paths, in_background=True)
print(f"Ruse process pid: {ruse_pid}")

In [ ]:
if check_process_running(ruse_pid):
    raise StopExecution

In [ ]:
metadata, tasks = parse_results("results/ruse_all_tasks_results", dry_run=False)
tasks.to_csv("all_tasks_results.csv", index=False)


In [ ]:
failed_tasks = tasks[tasks["found"].isnull()]

print("Total time for all tasks {}".format(metadata["total_time"]))
print(f"{len(failed_tasks)}/{len(tasks)} tasks failed")

failed_tasks.to_csv("failed_tasks.csv", index=False)
display_df(failed_tasks[["name", "path", "error"]])

In [ ]:
passed_tasks = tasks[tasks["found"].notnull()]
print(f"{len(passed_tasks)}/{len(tasks)} tasks passed")
display_df(passed_tasks[["name", "found"]])
